<center>
  <img src="../pics/rolling_hrz.png" width="300" />
</center>


- 任务状态分类
```txt
┌─────────────┬──────────────┬──────────────────────────────┐
│   状态       │   含义        │   动态调度时是否会被改动       │
├─────────────┼──────────────┼──────────────────────────────┤
│  RUNNING    │ 设备正在执行  │  ❌ 绝对不动（硬锁定）         │
│  SCHEDULED  │ 已下发待执行  │  ⚠️  看情况（软锁定）          │
│  PENDING    │ 尚未下发      │  ✅ 可以自由重排               │
└─────────────┴──────────────┴──────────────────────────────┘
```

- 时间轴示意
```txt
──────────────────────────────────────────────────────────
         已发生            当前时刻          未来
  ───────────┬───────────────┬──────────────────────────
             │               │
  [DONE][DONE][  RUNNING  ]  │  [SCHEDULED][PENDING][PENDING]
                             │
                         current_time
                             │
                        ◄────┤────────────────────────►
                          锁定区                可重排区
```

场景1: 任务正在执行中（RUNNING）
---
```txt
  P1_pipette: ████████████  ← 正在执行，绝对不会改动
                    ↑
              发生故障事件

  结论: ❌ 不受影响，继续执行到完成
```

场景2: 任务已下发但未开始（SCHEDULED）
---
```txt
  情况A: 该任务本身不受事件影响
         P1_load: [scheduled at t=7] → ✅ 保持不变

  情况B: 该任务依赖的资源发生故障
         P2_pipette: [scheduled, needs pipettor]
         pipettor 故障 → ⚠️  变为 BLOCKED，从时间表移除

  情况C: 前序任务提前完成
         P1_load: [scheduled at t=7]
         P1_pipette 提前完成 → ⏩ 提前到当前时刻开始
```

场景3: 任务尚未下发（PENDING）
---
```txt
  P2_incubate: [pending]
  → 战略层重规划时完全自由重排 ✅
```

为什么需要冻结窗口？
---

  问题: 如果允许调度系统频繁修改"即将开始"的任务，
        设备控制层会不断收到撤销/重发指令，造成混乱。

  解决: 设定冻结窗口，窗口内的已下发任务视同 RUNNING 处理。

```txt
  current_time          freeze_window
       │◄──────── 5min ────────►│
       │                        │
  ─────┼────────────────────────┼──────────────────────
       │   [  冻结区  ]          │    [ 可重排区 ]
       │                        │
  RUNNING / SCHEDULED        PENDING
  绝对不动                    自由重排
```

实现逻辑（战略层 _solve 内部）
---
```python
  FREEZE_WINDOW = 300  # 5分钟（秒）

  for task in all_tasks:
      if task.status == RUNNING:
          → committed（硬锁定）

      elif task.status == SCHEDULED:
          if task.start_time < current_time + FREEZE_WINDOW:
              → committed（软锁定，视同硬锁定）
          else:
              → free（可重排）

      elif task.status == PENDING:
          → free（可重排）
```

以"pipettor故障 → 恢复 → 重排"为例
---
```txt
t=0   战略层初始规划
      ┌────────────────────────────────────┐
      │ P1: pick→pipette→load→incubate→read│
      │ P2: pick→pipette→load→incubate→read│
      └────────────────────────────────────┘

t=3   pipettor 故障（规则层，<1ms）
      ┌────────────────────────────────────┐
      │ P2_pipette: SCHEDULED → BLOCKED    │  ← 规则层直接修改
      │ P2_load / incubate / read: 移出表  │
      │ P1 所有任务: 不受影响 ✅            │
      └────────────────────────────────────┘

t=8   pipettor 恢复（规则层，<1ms）
      ┌────────────────────────────────────┐
      │ P2_pipette: BLOCKED → PENDING      │  ← 规则层修改状态
      │ 调度表暂时没有 P2 的后续任务        │
      └────────────────────────────────────┘
      ↓ 立即触发战略层重规划（~50ms）
      ┌────────────────────────────────────┐
      │ P2_pipette 重新排入 t=8            │  ← CP-SAT 全局最优
      │ P2 后续任务重新排入                 │
      └────────────────────────────────────┘
```

两层分工
  规则层: 快速响应，保证系统不崩溃（<1ms）
  战略层: 慢速优化，保证全局最优（~50ms~500ms）


# DEMO

时间轴（横轴=时间单位）
```txt
────────────────────────────────────────────────────────────────────
t=  0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20
────────────────────────────────────────────────────────────────────

初始规划（CP-SAT 全局最优）
P1_pick      ██
P2_pick         ██
P1_pipette      █████
P2_pipette               █████
P1_load                  █
P2_load                           █
P1_incubate               ██████████
P2_incubate                        ██████████
P1_read                                      ███
P2_read                                               ███

────────────────────────────────────────────────────────────────────
t=3  事件1: pipettor 故障
────────────────────────────────────────────────────────────────────

  规则层响应（<1ms）：
    P1_pipette  RUNNING   → 不动 ✅
    P2_pipette  SCHEDULED → BLOCKED ⛔
    P2_load / P2_incubate / P2_read 级联移出调度表

  战略层重规划（~50ms）：
    pipettor broken → P2_pipette 无法排入
    重规划仅含 P1 剩余任务

  调度表变化：
  P1_pipette      ██?██   ← 正在执行，不动
  P2_pipette      -----   ← 移出
  P2_load         -----   ← 移出
  P2_incubate     -----   ← 移出
  P2_read         -----   ← 移出

────────────────────────────────────────────────────────────────────
t=5  事件2: P1_pipette 提前完成（原计划 t=7）
────────────────────────────────────────────────────────────────────

  规则层响应（<1ms）：
    P1_pipette  RUNNING → DONE ✅
    pipettor released（仍故障，available 不变）
    P1_load 前序全部 DONE → 提前到 t=5 开始

  调度表变化：
  P1_load     原: [7~8]  →  新: [5~6]   ⏩ 提前 2s
  P1_incubate 原: [8~18] →  新: [6~16]  ⏩ 连带提前

  无需战略层介入 ✅

────────────────────────────────────────────────────────────────────
t=8  事件3: pipettor 恢复
────────────────────────────────────────────────────────────────────

  规则层响应（<1ms）：
    pipettor available=1，broken=False
    P2_pipette: BLOCKED → PENDING

  战略层触发重规划（~50ms）：
    锁定任务: P1_incubate(RUNNING, [6~16])
    自由任务: P2_pipette / P2_load / P2_incubate / P2_read
              + P1_read (PENDING)

    资源约束:
      robot     容量=1  → P2_load 与其他 robot 任务不重叠
      pipettor  容量=1  → P2_pipette 独占
      incubator 容量=2  → P1_incubate + P2_incubate 可重叠 ✅
      reader    容量=1  → P1_read 和 P2_read 不重叠

  重规划结果：
  P2_pipette   [8~13]
  P2_load      [13~14]
  P2_incubate  [14~24]   ← 与 P1_incubate[6~16] 部分重叠（incubator 容量=2）
  P1_read      [16~19]
  P2_read      [24~27]

────────────────────────────────────────────────────────────────────
t=10  事件4: 紧急插入 P3_read（直接读板，无前序）
────────────────────────────────────────────────────────────────────

  规则层响应（<1ms）：
    reader 当前 available=1，not broken
    P3_read 贪心插入 t=10，状态 → SCHEDULED

  战略层触发重规划（~50ms）：
    P3_read 已占用 reader[10~12]
    P1_read 原计划[16~19]，reader 在 t=16 空闲 → 不冲突 ✅
    P2_read 原计划[24~27]，不冲突 ✅

  最终调度表：
  P3_read      [10~12]   ← 紧急插入
  P1_read      [16~19]   ← 不受影响
  P2_read      [24~27]   ← 不受影响

════════════════════════════════════════════════════════════════════

最终完整时间线
────────────────────────────────────────────────────────────────────
t=  0  2  4  5  6  8 10 12 13 14 16 18 19 24 27
────────────────────────────────────────────────────────────────────
P1_pick      ██
P2_pick         ██
P1_pipette      ████  (提前完成于t=5)
P2_pipette               █████
P1_load              █         (提前到t=5)
P2_load                           █
P1_incubate           ██████████
P2_incubate                       ██████████
P3_read                      ██            (紧急插入)
P1_read                                ███
P2_read                                              ███
────────────────────────────────────────────────────────────────────
makespan = 27s（若无故障原本 26s，故障恢复后 +1s）
```


# 简化DEMO
取板 → 加样 → 放入孵育箱 → 孵育 → 读板

双层动态调度：战略层(CP-SAT) + 规则层(事件响应)

In [14]:
import time
import threading
import queue
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from enum import Enum, auto
from collections import defaultdict
from ortools.sat.python import cp_model

In [15]:
# ============================================================
# 数据结构
# ============================================================

class Status(Enum):
    PENDING   = auto()   # 未下发，可重排
    SCHEDULED = auto()   # 已下发，原则不动
    RUNNING   = auto()   # 执行中，绝对不动
    DONE      = auto()   # 已完成
    BLOCKED   = auto()   # 资源故障，挂起


class EvType(Enum):
    FAILURE   = auto()   # 设备故障
    RECOVERY  = auto()   # 设备恢复
    COMPLETED = auto()   # 任务完成
    URGENT    = auto()   # 紧急插单


@dataclass
class Task:
    name:      str
    duration:  int
    res:       Dict[str, int]   # {资源: 需求}
    preds:     List[str]        # 前序任务名
    status:    Status           = Status.PENDING
    t_start:   Optional[float]  = None
    t_end:     Optional[float]  = None


@dataclass
class Resource:
    capacity:  int
    available: int
    broken:    bool = False


@dataclass
class Event:
    type:    EvType
    payload: Dict = field(default_factory=dict)


# 调度表：{任务名: (start, end)}
Schedule = Dict[str, Tuple[float, float]]

In [16]:
# ============================================================
# 战略层：CP-SAT 全局重规划
# ============================================================

def strategic_solve(
    tasks:     Dict[str, Task],
    resources: Dict[str, Resource],
    now:       float,
) -> Optional[Schedule]:
    """
    对 PENDING / SCHEDULED 任务做全局最优排布。
    RUNNING 任务作为固定区间锁定，不参与优化。
    """

    # 分类
    committed: Dict[str, Tuple] = {}   # {name: (start_offset, end_offset)}
    free:      List[Task]       = []

    for name, task in tasks.items():
        if task.status == Status.RUNNING:
            committed[name] = (task.t_start - now, task.t_end - now)
        elif task.status in (Status.PENDING, Status.SCHEDULED):
            free.append(task)

    if not free:
        return {}

    horizon = sum(t.duration for t in free) + 20
    m       = cp_model.CpModel()
    solver  = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 0.5
    solver.parameters.log_search_progress = False

    # 决策变量
    tv: Dict[str, Dict] = {}
    for t in free:
        s   = m.new_int_var(0, horizon,          f"s_{t.name}")
        e   = m.new_int_var(0, horizon + t.duration, f"e_{t.name}")
        itv = m.new_interval_var(s, t.duration, e, f"i_{t.name}")
        tv[t.name] = {"s": s, "e": e, "itv": itv}

    # 前序约束
    free_names = {t.name for t in free}
    done_names = {n for n, t in tasks.items() if t.status == Status.DONE}

    for t in free:
        for p in t.preds:
            if p in committed:
                m.add(tv[t.name]["s"] >= int(committed[p][1]))
            elif p in free_names:
                m.add(tv[t.name]["s"] >= tv[p]["e"])
            # DONE → 无约束；BLOCKED → 暂时忽略

    # 资源约束
    res_items: Dict[str, list] = defaultdict(list)

    for name, (cs, ce) in committed.items():
        task  = tasks[name]
        dur_c = max(1, int(ce - cs))
        sc    = m.new_constant(int(cs))
        ec    = m.new_constant(int(cs) + dur_c)
        ic    = m.new_interval_var(sc, dur_c, ec, f"c_{name}")
        for r, d in task.res.items():
            res_items[r].append((ic, d))

    for t in free:
        for r, d in t.res.items():
            res_items[r].append((tv[t.name]["itv"], d))

    for r_name, items in res_items.items():
        res = resources.get(r_name)
        if res is None:
            continue
        cap = 0 if res.broken else res.capacity
        m.add_cumulative([iv for iv, _ in items], [d for _, d in items], cap)

    # 目标
    makespan = m.new_int_var(0, horizon, "ms")
    m.add_max_equality(makespan, [v["e"] for v in tv.values()])
    m.minimize(makespan)

    status = solver.solve(m)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        return None

    result: Schedule = {}
    for name, v in tv.items():
        result[name] = (now + solver.value(v["s"]), now + solver.value(v["e"]))
    for name, (cs, ce) in committed.items():
        result[name] = (now + cs, now + ce)

    return result

In [17]:
# ============================================================
# 规则层：毫秒级事件响应
# ============================================================

def rule_handle(
    event:     Event,
    tasks:     Dict[str, Task],
    resources: Dict[str, Resource],
    schedule:  Schedule,
    now:       float,
) -> Tuple[Schedule, bool]:
    """
    返回 (修补后的schedule, 是否需要战略层重规划)
    """
    s = dict(schedule)

    # ── 设备故障 ─────────────────────────────────────────────
    if event.type == EvType.FAILURE:
        res_name = event.payload["resource"]
        resources[res_name].broken    = True
        resources[res_name].available = 0
        print(f"  [Rule] ⚠️  {res_name} 故障")

        for name, task in tasks.items():
            if res_name in task.res and task.status == Status.SCHEDULED:
                task.status = Status.BLOCKED
                s.pop(name, None)
                print(f"  [Rule] {name} → BLOCKED，移出调度表")

        return s, True   # 需要重规划

    # ── 设备恢复 ─────────────────────────────────────────────
    if event.type == EvType.RECOVERY:
        res_name = event.payload["resource"]
        res = resources[res_name]
        res.broken    = False
        res.available = res.capacity
        print(f"  [Rule] ✅  {res_name} 恢复")

        for name, task in tasks.items():
            if res_name in task.res and task.status == Status.BLOCKED:
                task.status = Status.PENDING
                print(f"  [Rule] {name} → PENDING，等待重排")

        return s, True   # 需要重规划

    # ── 任务完成 ─────────────────────────────────────────────
    if event.type == EvType.COMPLETED:
        name        = event.payload["task_name"]
        finish_time = event.payload.get("finish_time", now)
        task        = tasks[name]
        task.status = Status.DONE
        task.t_end  = finish_time
        print(f"  [Rule] ✔  {name} 完成 @ t={finish_time}")

        # 释放资源
        for r, d in task.res.items():
            resources[r].available = min(
                resources[r].capacity,
                resources[r].available + d,
            )

        # 后继提前
        for succ_name, succ in tasks.items():
            if name in succ.preds and succ.status == Status.SCHEDULED:
                all_done = all(
                    tasks[p].status == Status.DONE
                    for p in succ.preds if p in tasks
                )
                if all_done and succ_name in s:
                    old_s = s[succ_name][0]
                    if old_s > now:
                        s[succ_name] = (now, now + succ.duration)
                        print(f"  [Rule] ⏩ {succ_name} 提前: {old_s:.0f} → {now:.0f}")

        return s, False   # 规则层可自行处理

    # ── 紧急插单 ─────────────────────────────────────────────
    if event.type == EvType.URGENT:
        td   = event.payload["task"]
        name = td["name"]
        task = Task(
            name=name, duration=td["duration"],
            res=td.get("res", {}), preds=td.get("preds", []),
        )
        tasks[name] = task

        res_ok = all(
            not resources[r].broken and resources[r].available >= d
            for r, d in task.res.items() if r in resources
        )
        if res_ok:
            s[name]      = (now, now + task.duration)
            task.status  = Status.SCHEDULED
            task.t_start = now
            task.t_end   = now + task.duration
            print(f"  [Rule] 🚨 {name} 紧急插入 @ t={now:.0f}")
        else:
            print(f"  [Rule] 🚨 {name} 资源不足 → PENDING")

        return s, True   # 触发全局重排

    return s, False

In [18]:
# ============================================================
# 协调器
# ============================================================

class Scheduler:

    def __init__(self, tasks: Dict[str, Task], resources: Dict[str, Resource]):
        self.tasks     = tasks
        self.resources = resources
        self.now       = 0.0
        self.schedule: Schedule = {}
        self._lock     = threading.Lock()
        self._q: queue.Queue[Event] = queue.Queue()

    def start(self):
        # 初始规划
        self._replan("初始规划")
        # 事件循环线程
        threading.Thread(target=self._loop, daemon=True).start()

    def post(self, event: Event):
        self._q.put(event)

    def _loop(self):
        while True:
            try:
                ev = self._q.get(timeout=1.0)
            except queue.Empty:
                continue

            with self._lock:
                cur = dict(self.schedule)

            new_sched, need_replan = rule_handle(
                ev, self.tasks, self.resources, cur, self.now
            )

            with self._lock:
                self.schedule = new_sched

            if need_replan:
                self._replan(f"重规划（{ev.type.name}）")

    def _replan(self, label: str):
        t0    = time.perf_counter()
        sched = strategic_solve(self.tasks, self.resources, self.now)
        ms    = (time.perf_counter() - t0) * 1000

        if sched is not None:
            with self._lock:
                self.schedule = sched
            print(f"  [Strategic] {label}  耗时={ms:.1f}ms")
            self._print()

    def _print(self):
        s = self.schedule
        if not s:
            return
        max_e = max(int(e) for _, e in s.values())
        print(f"\n  {'任务':<22} {'start':>5} {'end':>5}  甘特图")
        print(f"  {'─'*22} {'─'*5} {'─'*5}  {'─'*max_e}")
        for name, (st, en) in sorted(s.items(), key=lambda x: x[1][0]):
            bar = "·" * int(st) + "█" * int(en - st) + "·" * (max_e - int(en))
            print(f"  {name:<22} {st:>5.0f} {en:>5.0f}  |{bar}|")
        print()

# Demo 场景


In [19]:
# ── 初始任务（P1、P2 两块板）────────────────────────────
tasks = {
    "P1_pick":     Task("P1_pick",     2, {"robot": 1},     []),
    "P1_pipette":  Task("P1_pipette",  5, {"pipettor": 1},  ["P1_pick"]),
    "P1_load":     Task("P1_load",     1, {"robot": 1},     ["P1_pipette"]),
    "P1_incubate": Task("P1_incubate", 10,{"incubator": 1}, ["P1_load"]),
    "P1_read":     Task("P1_read",     3, {"reader": 1},    ["P1_incubate"]),
    "P2_pick":     Task("P2_pick",     2, {"robot": 1},     []),
    "P2_pipette":  Task("P2_pipette",  5, {"pipettor": 1},  ["P2_pick"]),
    "P2_load":     Task("P2_load",     1, {"robot": 1},     ["P2_pipette"]),
    "P2_incubate": Task("P2_incubate", 10,{"incubator": 1}, ["P2_load"]),
    "P2_read":     Task("P2_read",     3, {"reader": 1},    ["P2_incubate"]),
}

resources = {
    "robot":     Resource(capacity=1, available=1),
    "pipettor":  Resource(capacity=1, available=1),
    "incubator": Resource(capacity=2, available=2),
    "reader":    Resource(capacity=1, available=1),
}

scheduler = Scheduler(tasks, resources)
scheduler.start()
time.sleep(0.3)

  [Strategic] 初始规划  耗时=143.4ms

  任务                     start   end  甘特图
  ────────────────────── ───── ─────  ──────────────────────────
  P2_pick                    0     2  |██························|
  P1_pick                    2     4  |··██······················|
  P2_pipette                 2     7  |··█████···················|
  P1_pipette                 7    12  |·······█████··············|
  P2_load                    7     8  |·······█··················|
  P2_incubate                8    18  |········██████████········|
  P1_load                   12    13  |············█·············|
  P1_incubate               13    23  |·············██████████···|
  P2_read                   18    21  |··················███·····|
  P1_read                   23    26  |·······················███|



In [20]:
# ── 事件1: t=3  pipettor 故障 ────────────────────────────
# P1_pipette 正在执行(RUNNING) → 不动
# P2_pipette 已下发(SCHEDULED) → BLOCKED，移出调度表
print("=" * 55)
print("事件1: t=3  pipettor 故障")
print("=" * 55)
scheduler.now = 3.0
tasks["P1_pick"].status  = Status.DONE
tasks["P1_pipette"].status  = Status.RUNNING
tasks["P1_pipette"].t_start = 2.0
tasks["P1_pipette"].t_end   = 7.0
tasks["P2_pick"].status  = Status.DONE
tasks["P2_pipette"].status  = Status.SCHEDULED
tasks["P2_pipette"].t_start = 7.0
tasks["P2_pipette"].t_end   = 12.0
scheduler.post(Event(EvType.FAILURE, {"resource": "pipettor"}))
time.sleep(0.5)

事件1: t=3  pipettor 故障
  [Rule] ⚠️  pipettor 故障
  [Rule] P2_pipette → BLOCKED，移出调度表


In [21]:
# ── 事件2: t=5  P1_pipette 提前完成 ─────────────────────
# 原计划 t=7 完成，实际 t=5 完成
# 规则层直接将 P1_load 提前到 t=5，无需重规划
print("=" * 55)
print("事件2: t=5  P1_pipette 提前完成（原计划 t=7）")
print("=" * 55)
scheduler.now = 5.0
scheduler.post(Event(EvType.COMPLETED, {
    "task_name": "P1_pipette", "finish_time": 5.0
}))
time.sleep(0.3)

事件2: t=5  P1_pipette 提前完成（原计划 t=7）
  [Rule] ✔  P1_pipette 完成 @ t=5.0


In [22]:
 # ── 事件3: t=8  pipettor 恢复 ───────────────────────────
# P2_pipette: BLOCKED → PENDING
# 战略层重新排入 P2 全链路，与 P1 剩余任务共同全局优化
print("=" * 55)
print("事件3: t=8  pipettor 恢复")
print("=" * 55)
scheduler.now = 8.0
tasks["P1_load"].status    = Status.DONE
tasks["P1_load"].t_end     = 6.0
tasks["P1_incubate"].status  = Status.RUNNING
tasks["P1_incubate"].t_start = 6.0
tasks["P1_incubate"].t_end   = 16.0
scheduler.post(Event(EvType.RECOVERY, {"resource": "pipettor"}))
time.sleep(0.5)

事件3: t=8  pipettor 恢复
  [Rule] ✅  pipettor 恢复
  [Rule] P2_pipette → PENDING，等待重排
  [Strategic] 重规划（RECOVERY）  耗时=138.4ms

  任务                     start   end  甘特图
  ────────────────────── ───── ─────  ───────────────────────────
  P1_incubate                6    16  |······██████████···········|
  P2_pipette                 8    13  |········█████··············|
  P2_load                   13    14  |·············█·············|
  P2_incubate               14    24  |··············██████████···|
  P1_read                   16    19  |················███········|
  P2_read                   24    27  |························███|



In [23]:
# ── 事件4: t=10  紧急插入 P3_read ───────────────────────
# reader 当前空闲 → 贪心插入 t=10
# 战略层重规划确保 P1_read / P2_read 不与 P3_read 冲突
print("=" * 55)
print("事件4: t=10  紧急插入 P3_read")
print("=" * 55)
scheduler.now = 10.0
scheduler.post(Event(EvType.URGENT, {"task": {
    "name": "P3_read", "duration": 2,
    "res":  {"reader": 1}, "preds": [],
}}))
time.sleep(0.5)

事件4: t=10  紧急插入 P3_read
  [Rule] 🚨 P3_read 紧急插入 @ t=10


In [24]:
# ── 最终结果 ─────────────────────────────────────────────
print("=" * 55)
print("最终调度方案")
print("=" * 55)
scheduler._print()

最终调度方案

  任务                     start   end  甘特图
  ────────────────────── ───── ─────  ───────────────────────────
  P1_incubate                6    16  |······██████████···········|
  P2_pipette                 8    13  |········█████··············|
  P3_read                   10    12  |··········██···············|
  P2_load                   13    14  |·············█·············|
  P2_incubate               14    24  |··············██████████···|
  P1_read                   16    19  |················███········|
  P2_read                   24    27  |························███|




g_i(x) \le 0,\quad h_j(x)=0

$$
\begin{aligned}
\text{s.t.}\quad 
& g_i(x) \le 0, \quad i=1,2,\ldots,m, \\
& h_j(x) = 0, \quad j=1,2,\ldots,p.
\end{aligned}
$$